In [47]:
import jax
import jax.numpy as jnp
import optax
import flax.linen as nn
from flax.training import train_state
from flax.training.common_utils import onehot
from flax.training import checkpoints
from flax import jax_utils
from flax.core import freeze, unfreeze
import numpy as np
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import math
from itertools import islice, cycle



In [ ]:
# Don't use TFDS because SSL sometimes is dumb on site; use Jax loader instead
# https://github.com/jax-ml/jax/blob/main/examples/datasets.py

import sys
sys.path.append('../utils')

import datasets
train_images, train_labels, test_images, test_labels = datasets.mnist()
train_images, train_labels, test_images, test_labels = jnp.array(train_images), jnp.array(train_labels), jnp.array(test_images), jnp.array(test_labels)

# Create a generator that yields shuffled batches of data without repetition until an epoch is complete
def data_generator(images, labels, batch_size):
    num_samples = images.shape[0]
    # while True:
    #     # Shuffle the data at the beginning of each epoch
    indices = np.random.permutation(num_samples)
    for start_idx in range(0, num_samples, batch_size):
        end_idx = min(start_idx + batch_size, num_samples)
        batch_indices = indices[start_idx:end_idx]
        yield {'image': images[batch_indices], 'label': labels[batch_indices]}
            

In [51]:
train_ds = data_generator(train_images[0:4], train_labels[0:4], 1)
for batch in islice(cycle(train_ds), 8):
    print(batch['image'].shape)


(1, 784)
(1, 784)
(1, 784)
(1, 784)
(1, 784)
(1, 784)
(1, 784)
(1, 784)


In [36]:
# Model definition (Linen)
class MLP(nn.Module):
    depth: int
    width: int
    activation: str

    @nn.compact
    def __call__(self, x):
        x = x.reshape(x.shape[0], -1)  # Flatten
        for i in range(self.depth):
            if i == 0:
                x = nn.Dense(self.width)(x)
            elif i == self.depth - 1:
                x = nn.Dense(10)(x)
            else:
                x = nn.Dense(self.width)(x)
            if i < self.depth - 1:
                x = nn.relu(x)
        return x



In [37]:

def create_train_state(rng, model, learning_rate, weight_decay):
    params = model.init(rng, jnp.ones([1, 784]))['params']
    tx = optax.adamw(learning_rate, weight_decay=weight_decay)
    return train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)



In [ ]:

### Choose experiment parameters
train_points = 1000
optimization_steps = 100000
batch_size = 200
weight_decay = 0.01
lr = 1e-3
initialization_scale = 8.0
download_directory = "."

depth = 3               # the number of nn.Linear modules in the model
width = 200
activation = 'relu'     # 'relu' or 'tanh' or 'sigmoid' or 'gelu'

log_freq = math.ceil(optimization_steps / 150)
rng = jax.random.PRNGKey(0)

# Create model
model = MLP(depth=depth, width=width, activation=activation)
state = create_train_state(rng, model, lr, weight_decay)

train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []
norms = []
last_layer_norms = []
log_steps = []

train_ds = data_generator(train_images[0:train_points], train_labels[0:train_points], batch_size)
test_ds = data_generator(test_images, test_images, batch_size)

steps = 0
with tqdm(total=optimization_steps) as pbar:

    @jax.jit
    def mse_loss(logits, labels):
        return jnp.mean(jnp.sum((logits - labels) ** 2, axis=-1))
    
    @jax.jit
    def loss_fn_wrapper(params):
        logits = model.apply({'params': params}, x)
        loss = mse_loss(logits, labels)
        return loss

    grad_fn = jax.value_and_grad(loss_fn_wrapper)
    
    for batch in islice(cycle(train_ds), optimization_steps):
        x, labels = batch['image'], batch['label']
        @jax.jit
        def batch_loss_accuracy(params, x, labels):
            # Eval model 
            logits = model.apply({'params': params}, x)
            loss = mse_loss(logits, labels)

            # Get accuracy 
            predicted_labels = jnp.argmax(logits, axis=1) # shape (50,)
            correct = jnp.sum(predicted_labels == labels @ jnp.arange(10)) # convert labels (50, 10) to (50,)

            return  loss, correct

        def compute_loss_and_accuracy(params, dataset):
            """Computes mean loss of `model` on `dataset`."""
            total_loss = 0
            total_points = 0
            correct = 0

            for batch in dataset:
                x, labels = batch['image'], batch['label']
            
                batch_loss, batch_accuracy = batch_loss_accuracy(params, x, labels)

                total_loss += batch_loss
                correct += batch_accuracy
                total_points += x.shape[0]

            return total_loss / total_points, (correct / total_points)
        
        loss, grads = grad_fn(state.params)
        state = state.apply_gradients(grads=grads)
        if (steps < 30) or (steps < 150 and steps % 10 == 0) or steps % log_freq == 0:
            epoch_loss, epoch_acc = compute_loss_and_accuracy(
                    state.params, data_generator(train_images[0:100], train_labels[0:100], batch_size=50)
            )
            train_losses.append(
                epoch_loss
            )
            train_accuracies.append(
                epoch_acc
            )

            epoch_loss, epoch_acc = compute_loss_and_accuracy(
                    state.params, data_generator(test_images, test_labels, batch_size=50)
            )
            test_losses.append(
                epoch_loss
            )
            test_accuracies.append(
                epoch_acc
            )

            log_steps.append(steps)
            norms.append(jnp.linalg.norm(jax.tree_util.tree_leaves(state.params)))
            last_layer_norms.append(jnp.linalg.norm(state.params['Dense_2']['kernel']))
            pbar.set_description("L: {0:1.1e}|{1:1.1e}. A: {2:2.1f}%|{3:2.1f}%".format(
                train_losses[-1],
                test_losses[-1],
                train_accuracies[-1] * 100, 
                test_accuracies[-1] * 100))

        steps += 1
        pbar.update(1)

ax = plt.subplot(1, 1, 1)
plt.plot(log_steps, train_accuracies, color='red', label='train')
plt.plot(log_steps, test_accuracies, color='green', label='test')
plt.xscale('log')
plt.xlim(10, None)
plt.xlabel("Optimization Steps")
plt.ylabel("Accuracy")
plt.legend(loc=(0.015, 0.75))

ax2 = ax.twinx()
ax2.set_ylabel("Weight Norm", color='purple')
ax2.plot(log_steps, norms, color='purple', label='weight norm')
plt.legend(loc=(0.015, 0.65))
plt.title(f"depth-3 width-200 ReLU MLP on MNIST\nUnconstrained Optimization α = {initialization_scale}", fontsize=11)
plt.show()

  0%|          | 0/100000 [00:00<?, ?it/s]

(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
(200, 784)
